# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeerMusabih/FlyRank-AI-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I selected a Random Forest classifier for this modeling task.

The goal of this lane is to prioritize content pages for refresh review. The Week-4 baseline uses a simple rule-based score, while the learned model can combine several observed content, traffic, freshness, and engagement signals.

Random Forest fits the task because it can capture non-linear relationships and interactions between these structured signals while still providing feature importance for interpretation. It is also a reasonable next step from a simple rule baseline because the purpose is to test whether a learned model provides a useful improvement rather than adding complexity for its own sake.

The target is `is_declining_label`, which indicates whether the observed trend direction is down.

To avoid leakage, `trend_direction` and `trend_pct` are excluded. The 30-day comparison columns used to construct the trend label are also excluded because they directly contain the information used to create the target.

The model is used as decision support: its probability of decline is used to rank pages for review rather than automatically deciding that a page must be refreshed.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../../data/raw/content_refresh_anonymized.csv'

### Target and leakage checks

The target is `is_declining_label`.

The following fields are not used as model features:

- `is_declining_label` — target
- `trend_direction` — source of the target
- `trend_pct` — directly used to define the trend direction
- `impressions_last_30d`
- `clicks_last_30d`
- `sessions_last_30d`
- `impressions_prev_30d`
- `clicks_prev_30d`
- `sessions_prev_30d`

The six comparison-window fields are excluded because the target is constructed from the recent-vs-previous impression trend. Including them would allow the model to learn the label from the same information used to create it.

Identifiers are also excluded from the feature matrix. `client_id` is retained separately for grouped validation.

In [ ]:
TARGET = "is_declining_label"

ID_COLUMNS = [
    "content_id",
    "client_id"
]

LEAKAGE_COLUMNS = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

# The data dictionary explicitly says these should not be model features.
NON_FEATURE_COLUMNS = [
    "provider_used",
    "model_used"
]

excluded_columns = (
    [TARGET]
    + ID_COLUMNS
    + LEAKAGE_COLUMNS
    + NON_FEATURE_COLUMNS
)

feature_columns = [
    col for col in df.columns
    if col not in excluded_columns
]

X = df[feature_columns].copy()
y = df[TARGET].astype(int).copy()
groups = df["client_id"].copy()

print("Number of features:", len(feature_columns))
print("Target distribution:")
print(y.value_counts())
print("\nTarget rate:")
print(y.mean())

## 2. Split design

I use a client-grouped train/test split.

The dataset contains multiple content pages from the same pseudonymized client. A random row-level split could place pages from the same client in both training and testing, allowing client-specific patterns to appear in both sets.

Instead, `client_id` is used only for grouping. Entire clients are assigned to either the training or test set.

This gives a more honest estimate of how the model performs on clients it did not see during training.

The split is fixed with `random_state=42` so the experiment is reproducible.

In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

print("\nClient overlap:")
print(set(groups_train).intersection(set(groups_test)))

print("\nTraining target rate:", round(y_train.mean(), 4))
print("Test target rate:", round(y_test.mean(), 4))

## 3. Train + compare vs my baseline

The Week-4 baseline is the rule-based refresh score from ML-07:

- +2 for content that has not been updated for at least 180 days
- +1 for at least 500 impressions in the last 30 days
- +1 for average position greater than 10

Higher scores produce higher refresh priority.

Because this baseline produces a ranked queue rather than a binary prediction, the comparison uses Precision@20 and Precision@50. These metrics ask how many of the highest-priority pages are actually observed as declining.

The Random Forest produces a probability of decline, which is also used as a ranking score.

Both methods are evaluated on the same held-out test pages.

In [ ]:
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

In [ ]:
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median", add_indicator=True)
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced"
)

rf_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        model
    )
])

rf_pipeline.fit(X_train, y_train)

rf_probability = rf_pipeline.predict_proba(
    X_test
)[:, 1]

print("Model trained successfully.")

In [ ]:
model_results = df.iloc[test_idx][
    ["content_id", "client_id"]
].copy()

model_results["actual"] = y_test.values
model_results["model_score"] = rf_probability

model_results = model_results.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

model_results.head(10)

In [ ]:
baseline_test = df.iloc[test_idx].copy()

stale = (
    baseline_test["days_since_last_update"] >= 180
).astype(int)

visible = (
    baseline_test["impressions_last_30d"] >= 500
).astype(int)

low_position = (
    baseline_test["avg_position"] > 10
).astype(int)

baseline_test["baseline_score"] = (
    stale * 2 +
    visible +
    low_position
)

baseline_results = baseline_test[
    ["content_id", "client_id", TARGET, "baseline_score"]
].copy()

baseline_results = baseline_results.rename(
    columns={TARGET: "actual"}
)

baseline_results = baseline_results.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline_results.head(10)

In [ ]:
def precision_at_k(ranked_df, k):
    top_k = ranked_df.head(k)
    return top_k["actual"].mean()


ks = [20, 50]

comparison_rows = []

for k in ks:
    comparison_rows.append({
        "Metric": f"Precision@{k}",
        "ML-07 Baseline": precision_at_k(
            baseline_results,
            k
        ),
        "Random Forest": precision_at_k(
            model_results,
            k
        )
    })

comparison = pd.DataFrame(comparison_rows)

comparison

In [ ]:
base_rate = y_test.mean()

print(
    f"Test-set declining base rate: "
    f"{base_rate:.3f} ({base_rate:.1%})"
)

### Baseline vs model

The comparison table above uses the same held-out test pages for both approaches.

Precision@K is interpreted as the proportion of pages in the top K ranked recommendations that have the observed declining label.

The test-set base rate is shown as context. A useful ranking should be interpreted relative to this observed rate rather than by looking at the precision value alone.

The Random Forest is considered useful only if its ranking provides a meaningful improvement over the simple ML-07 rule, especially at the review sizes that matter for decision support.

## 4. Errors and interpretation

I inspect both model errors and feature importance before deciding whether the model is useful.

For a ranking model, an important error is a page that receives a high predicted decline probability but is not observed as declining. The opposite error is a page that is observed as declining but receives a relatively low score.

Three concrete test-set errors are inspected below rather than relying only on aggregate metrics.

In [ ]:
error_analysis = df.iloc[test_idx][
    [
        "content_id",
        "client_id",
        "is_declining_label",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "ctr",
        "avg_position"
    ]
].copy()

error_analysis["model_score"] = rf_probability

error_analysis["predicted"] = (
    error_analysis["model_score"] >= 0.5
).astype(int)

false_positives = error_analysis[
    (error_analysis["predicted"] == 1) &
    (error_analysis["is_declining_label"] == 0)
].sort_values(
    "model_score",
    ascending=False
)

false_negatives = error_analysis[
    (error_analysis["predicted"] == 0) &
    (error_analysis["is_declining_label"] == 1)
].sort_values(
    "model_score",
    ascending=True
)

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

In [ ]:
three_errors = pd.concat([
    false_positives.head(2),
    false_negatives.head(1)
])

three_errors[
    [
        "content_id",
        "is_declining_label",
        "model_score",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "ctr",
        "avg_position"
    ]
]

In [ ]:
fitted_preprocessor = rf_pipeline.named_steps["preprocessor"]
fitted_model = rf_pipeline.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()

importances = pd.Series(
    fitted_model.feature_importances_,
    index=feature_names
).sort_values(
    ascending=False
)

top_features = importances.head(10)

top_features

In [ ]:
import matplotlib.pyplot as plt

top_features.sort_values().plot.barh(
    figsize=(8, 5)
)

plt.title("Top Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

### Feature interpretation

The three strongest observed features in the fitted model were:

1. **[FEATURE 1]** — [brief explanation of why this signal could plausibly relate to observed decline].
2. **[FEATURE 2]** — [brief explanation].
3. **[FEATURE 3]** — [brief explanation].

These associations should not be interpreted as causal effects. They indicate which observed signals the model relied on when separating the two target classes.

I also checked the feature list for suspicious identifiers or direct label fields. Client and content identifiers were excluded, and the trend label/source fields were excluded from the model features.

### Error interpretation

The three inspected errors show that the classes are not perfectly separable from the available observed signals.

The false-positive examples received relatively high model scores but were not labeled as declining. This suggests that their observed traffic, engagement, freshness, or ranking characteristics resembled pages that were declining.

The false-negative example was observed as declining but received a lower model score. This indicates that some declining pages have signals that resemble the non-declining group.

These errors are important because they show that the model should be treated as decision support rather than an automatic refresh decision.

### Finding

The Random Forest should only be considered an improvement if its measured Precision@20 and/or Precision@50 is higher than the ML-07 baseline on the same held-out clients.

If the model improves the ranking at one review size but not another, that difference is the finding rather than something to hide.

The results therefore support a cautious conclusion about whether the learned model adds value over the simple rule-based baseline. They do not establish that the model will cause better SEO outcomes or that any individual page should definitely be refreshed.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.